In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/synthetic/occupancy_modeling_dataset.csv")
df.head()

,arrivals,depatures,occupancy,timestamp,hour,day_of_week,day_of_month,month,is_weekend,is_business_hour,...,occupancy_lag_8,occupancy_lag_96,occupancy_lag_672,rolling_mean_1h,rolling_mean_3h,rolling_mean_24h,rolling_std_1h,target_30min,target_60min,target_120min
0,0,0,1,2023-08-14 12:15:00,12,0,14,8,0,1,...,1.0,0.0,1.0,1.0,1.0,0.145833,0.354792,1.0,1.0,1.0
1,0,0,1,2023-08-14 12:30:00,12,0,14,8,0,1,...,1.0,0.0,1.0,1.0,1.0,0.156250,0.364998,1.0,1.0,1.0
2,0,0,1,2023-08-14 12:45:00,12,0,14,8,0,1,...,1.0,0.0,1.0,1.0,1.0,0.166667,0.374634,1.0,1.0,1.0
3,0,0,1,2023-08-14 13:00:00,13,0,14,8,0,1,...,1.0,0.0,1.0,1.0,1.0,0.177083,0.383743,1.0,1.0,1.0
4,0,0,1,2023-08-14 13:15:00,13,0,14,8,0,1,...,1.0,0.0,1.0,1.0,1.0,0.187500,0.392361,1.0,1.0,1.0


In [2]:
print("Dataset shape:", df.shape)

df['timestamp'] = pd.to_datetime(df['timestamp'])

print("\nTime range:")
print("Start:", df["timestamp"].min())
print("End:",  df["timestamp"].max())

print("\nTime interval distribution:")
print(
    df["timestamp"]
    .diff()
    .value_counts()
    .head(10)
)

print("\nMissing intervals:")
expected_intervals = pd.Timedelta(minutes=15)

gaps = (
    df["timestamp"].diff() > expected_intervals
)

print("Number of gaps:", gaps.sum())

Dataset shape: (25768, 27)

Time range:
Start: 2023-08-14 12:15:00
End: 2024-05-08 22:00:00

Time interval distribution:
timestamp
0 days 00:15:00    25767
Name: count, dtype: int64

Missing intervals:
Number of gaps: 0


In [3]:
feature_columns = [
    "occupancy",
    "arrivals",
    "depatures",
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",
    "is_business_hour",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "occupancy_lag_1",
    "occupancy_lag_2",
    "occupancy_lag_4",
    "occupancy_lag_8",
    "occupancy_lag_96",
    "occupancy_lag_672",
    "rolling_mean_1h",
    "rolling_mean_3h",
    "rolling_mean_24h",
    "rolling_std_1h"
]

target_columns = [
    "target_30min",
    "target_60min",
    "target_120min"
]

X = df[feature_columns]

y = df[target_columns]

print("Input features:", X.shape)
print("Prediction targets:", y.shape)

Input features: (25768, 23)
Prediction targets: (25768, 3)


In [5]:
df = df.sort_values(by="timestamp").reset_index(drop=True)

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]

X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining period")
print(df["timestamp"].iloc[0])
print(df["timestamp"].iloc[split_index - 1])

print("\nTesting period")
print(df["timestamp"].iloc[split_index])
print(df["timestamp"].iloc[-1])


Training records: 20614
Testing records: 5154

Training period
2023-08-14 12:15:00
2024-03-16 05:30:00

Testing period
2024-03-16 05:45:00
2024-05-08 22:00:00


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

baseline_results = []

for target in target_columns:

    actual = y_test[target]

    
    predicted = X_test["occupancy"]

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = r2_score(actual, predicted)

    baseline_results.append({
        "Target": target,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

baseline_df = pd.DataFrame(baseline_results)

baseline_df

,Target,MAE,RMSE,R2
0,target_30min,7.925301,13.507748,0.955421
1,target_60min,14.566162,24.599217,0.852171
2,target_120min,26.575669,43.378779,0.540459
